In [ ]:
import pymupdf
from pathlib import Path
import re

def pdf_to_markdown(pdf_path, markdown_path):
    markdown = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")
            
    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))
    print(f"Markdown file created: {markdown_path}")

# الاستدعاء بمسار مجلد المشروع الرئيسي
pdf_to_markdown("../Assets/harrypotter.pdf", "../output.md")

In [ ]:


# 2. Split into dataset pages
INPUT_FILE = Path("../output.md")
OUTPUT_FOLDER = Path("../dataset")

BOOK_RANGES = [
    ("Harry Potter and the Sorcerer Stone", 12, 274),
    ("Harry Potter and the Chamber of Secrets", 282, 565),
    ("Harry Potter and the Prisoner of Azkaban", 573, 939),
    ("Harry Potter and the Goblet of Fire", 949, 1560),
    ("Harry Potter and the Order of the Phoenix", 1570, 2406),
    ("Harry Potter and the Half-Blood Prince", 2409, 2964),
    ("Harry Potter and the Deathly Hallows", 2974, 3622),
]

def get_book_name(page_number):
    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name
    return None

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.replace(r'[\r\n]+', '\n').strip()
    return text

def split_pages():
    if not INPUT_FILE.exists():
        print("output.md not found! Skip splitting or provide output.md")
        return
    text = INPUT_FILE.read_text(encoding="utf-8")
    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)
    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):
        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_book_name(page_number)

        if book_name:
            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )
            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")

split_pages()

In [ ]:
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

load_dotenv(dotenv_path="../backend/.env")

DATASET_FOLDER = Path("../dataset")
MODEL_NAME = "intfloat/multilingual-e5-large"

def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()
    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()
    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }

files = sorted(DATASET_FOLDER.glob("*.md"))
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

if texts:
    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()

    QDRANT_URL = os.getenv("QDRANT_URL")
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
    QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "harry_potter")

        
    print("QDRANT_URL:", os.getenv("QDRANT_URL"))
    print("QDRANT_API_KEY:", os.getenv("QDRANT_API_KEY"))

    
    client = QdrantClient(
        url=os.getenv("QDRANT_URL"),
        api_key=os.getenv("QDRANT_API_KEY"),
        timeout=120.0
    )
    vector_size = len(embeddings[0])

    if not client.collection_exists(QDRANT_COLLECTION):
        client.create_collection(
            collection_name=QDRANT_COLLECTION,
            vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
        )

    points = [
        models.PointStruct(id=index, vector=embedding, payload=page)
        for index, (page, embedding) in enumerate(zip(pages, embeddings))
    ]

    batch_size = 100
    for start in range(0, len(points), batch_size):
        batch = points[start:start + batch_size]
        client.upsert(collection_name=QDRANT_COLLECTION, points=batch)

    print(f"Uploaded {len(points)} pages to Qdrant.")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

query = "Who rescued Harry using a flying car?"

router_llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

SYSTEM_PROMPT = """You classify messages for a Harry Potter book search system.
Return exactly one label and nothing else:
retrieve - questions about the books, characters, places, or events
chitchat - greetings, thanks, or casual conversation
off-topic - anything unrelated to the books"""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]

route = router_llm.invoke(router_messages).content.strip().lower()
route = route.splitlines()[0].strip(" `.,:")
if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)

In [ ]:
top_k = 3

if route == "retrieve" and 'model' in locals():
    query_vector = model.encode([f"query: {query}"], normalize_embeddings=True)[0].tolist()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
    ).points

    for result in results:
        page = result.payload
        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"][:150], "...")
        print("-" * 60)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

evaluation_cases = [
    {
        "query": "Who rescued Harry from his locked bedroom using a flying car?",
        "relevant_pages": {301, 302},
    },
    {
        "query": "What loophole did Mr Weasley write into the law about enchanting a car?",
        "relevant_pages": {314},
    },
]

gemini_llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

for case in evaluation_cases:
    query_vector = model.encode([f"query: {case['query']}"], normalize_embeddings=True)[0].tolist()
    search_results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    ).points

    context = "\n\n".join(
        f"Page {result.payload['page_number']}: {result.payload['content']}"
        for result in search_results
    )

    answer = gemini_llm.invoke([
        SystemMessage(content="Answer only from the provided context. If the answer is not there, say you do not know."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}"),
    ]).text

    judge = gemini_llm.invoke([
        SystemMessage(content="""You are an evaluator for a question-answering system.
Judge the answer using only the context.
Return format:
Score: X/5
Grounded: yes or no
Reason: one short sentence"""),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}\n\nAnswer:\n{answer}"),
    ]).text

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:\n", judge)
    print("=" * 60)